# Notebook 03 — XML Cross-Border Partner Transactions Ingestion & EDA

**Source:** `ewallet_transactions.xml` (Bilpay E-Wallet, 28,360 records)  
**Owner:** Firas (original) | Consolidated by Elie Khairallah  
**Rubric:** §4.1, §4.3, §4.4

This notebook ingests the XML cross-border partner feed, validates it against the batch header, applies cleaning (including UUID regeneration and anomaly flagging), and produces exploratory visualisations.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
Path('data/audit').mkdir(exist_ok=True)
print('Working directory:', os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded')

## 1. Ingestion

The XML has a nested structure: `<batch> → <batch_header>` (metadata) + `<transactions> → <transaction>` (records). Financial amounts and transaction type are stored in sub-elements and attributes, not direct text nodes.

In [ ]:
from src.ingestion import ingest_xml
df_raw, batch_meta = ingest_xml.load_raw()
print('Batch metadata:')
for k, v in batch_meta.items():
    print(f'  {k}: {v}')
print(f'\nDataFrame shape: {df_raw.shape}')

In [ ]:
print('Columns:', df_raw.columns.tolist())
print('\nNull counts:')
print(df_raw.isnull().sum())
print(f'\nUnique wallets: {df_raw["wallet_id"].nunique()}')
print(f'Duplicate receipt_numbers (pre-clean): {df_raw["receipt_number"].duplicated().sum()}')

## 2. Data Quality Assessment (Pre-Cleaning)

Key issues found in the raw data:

In [ ]:
print('Transaction type distribution:')
print(df_raw['transaction_type'].value_counts())

print('\nReceipt number duplicates by type:')
dup_mask = df_raw['receipt_number'].duplicated(keep=False)
print(df_raw[dup_mask]['transaction_type'].value_counts())

c = pd.to_datetime(df_raw['created_at'])
p = pd.to_datetime(df_raw['paying_at'])
print(f'\nDate order anomalies (created_at > paying_at): {(c > p).sum():,}')

**Pre-cleaning findings:**
- All 15,060 TOP_UP rows share duplicated `receipt_number` values (design flaw — batch-generated placeholder UUIDs)
- 2,432 rows have `created_at > paying_at` (system clock skew at point of sale)
- `note` and `detail` fields are distinct (291 and 504 unique values) — both retained

## 3. Cleaning

In [ ]:
from src.cleaning import clean_xml
df_clean, report = clean_xml.clean(df_raw)
print('Cleaning actions:')
for a in report['actions']:
    print(f'  • {a}')
print(f'\nreceipt_number dups after cleaning: {df_clean["receipt_number"].duplicated().sum()}')
print(f'date_order_anomaly rows: {df_clean["date_order_anomaly"].sum()}')
print(f'is_ref_dup rows: {df_clean["is_ref_dup"].sum()}')

## 4. EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
types = df_clean['transaction_type'].value_counts()
types.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Transaction Type Distribution', fontweight='bold')
axes[0].set_xlabel('Type'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

channels = df_clean['channel'].value_counts()
channels.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='black')
axes[1].set_title('Channel Distribution', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
fig.savefig('data/audit/fig_07_xml_types_channels.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation:** TOP_UP dominates (53%) — this cross-border feed primarily serves as a wallet-funding channel for the Nigerian operator. ATM and mobile-banking channels account for most activity, consistent with Bilpay's Indonesian market positioning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
amt = df_clean['net_amount'].astype(float)
amt.plot(kind='hist', bins=50, ax=axes[0], color='steelblue', edgecolor='black', log=True)
axes[0].set_title('net_amount Distribution (log scale)', fontweight='bold')
axes[0].set_xlabel('Amount (IDR)')

df_clean['paying_at'].dt.to_period('M').astype(str).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='teal', edgecolor='black')
axes[1].set_title('Monthly Transaction Volume', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
fig.savefig('data/audit/fig_08_xml_amounts_monthly.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
anom_monthly = df_clean[df_clean['date_order_anomaly']].groupby(
    df_clean.loc[df_clean['date_order_anomaly'], 'paying_at'].dt.to_period('M').astype(str)
).size()
anom_monthly.plot(kind='bar', ax=ax, color='red', alpha=0.7, edgecolor='black')
ax.set_title('Date Order Anomalies by Month (created_at > paying_at)', fontweight='bold')
ax.set_ylabel('Anomaly Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
fig.savefig('data/audit/fig_09_xml_anomalies.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation:** Date order anomalies cluster in certain months, suggesting periodic batch-upload issues in the Bilpay system where transaction creation timestamps are recorded after payment timestamps. These rows are retained with the `date_order_anomaly` flag for analyst review.

## 5. Save Processed Output

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
Path('data/processed').mkdir(exist_ok=True)
Path('data/samples').mkdir(exist_ok=True)

df_clean.to_parquet('data/processed/xml_final.parquet', index=False)
print('Saved: data/processed/xml_final.parquet')

# Small XML sample for git
tree = ET.parse('data/raw/ewallet_transactions.xml')
root = tree.getroot()
txns = root.find('transactions')
sample_root = ET.Element('batch')
sample_root.append(root.find('batch_header'))
sample_txns = ET.SubElement(sample_root, 'transactions')
for tx in list(txns)[:100]:
    sample_txns.append(tx)
ET.ElementTree(sample_root).write('data/samples/xml_sample_100.xml', encoding='unicode', xml_declaration=True)
print('Saved: data/samples/xml_sample_100.xml')
print(f'\nFinal shape: {df_clean.shape}')